# classification assignment


In [6]:
# import libraries
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression



In [2]:
#load the dataset
df=pd.read_csv('../data/course_lead_scoring_2026.csv')
df.head()


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [4]:
# find the mode in industry column
industry_mode = df['industry'].mode()[0]
print(f"The mode of the 'industry' column is: {industry_mode}")

The mode of the 'industry' column is: technology


In [ ]:




categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.converted.values
y_val = df_val.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [8]:
corr = df_train[numerical].corr()
print(corr)

                          number_of_courses_viewed  annual_income  \
number_of_courses_viewed                  1.000000       0.166168   
annual_income                             0.166168       1.000000   
interaction_count                         0.714902       0.117885   
lead_score                                0.750809       0.221734   

                          interaction_count  lead_score  
number_of_courses_viewed           0.714902    0.750809  
annual_income                      0.117885    0.221734  
interaction_count                  1.000000    0.916535  
lead_score                         0.916535    1.000000  


In [9]:
for a, b in [('interaction_count', 'lead_score'),
             ('number_of_courses_viewed', 'lead_score'),
             ('number_of_courses_viewed', 'interaction_count'),
             ('annual_income', 'interaction_count')]:
    print(a, b, round(df_train[a].corr(df_train[b]), 3))

interaction_count lead_score 0.917
number_of_courses_viewed lead_score 0.751
number_of_courses_viewed interaction_count 0.715
annual_income interaction_count 0.118


In [10]:
for c in categorical:
    print(c, round(mutual_info_score(df_train[c], y_train), 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


In [11]:
features = categorical + numerical

def train_and_score(features, C=1.0, rounding=None):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    acc = accuracy_score(y_val, model.predict(X_val))
    return acc

acc = train_and_score(features)
print(round(acc, 2))

0.65


In [14]:
base_acc = train_and_score(features)   

for f in features:
    subset = [x for x in features if x != f]
    acc = train_and_score(subset)
    print(f, round(base_acc - acc, 5))

lead_source 0.003
industry 0.0
employment_status 0.0
location 0.0
number_of_courses_viewed 0.002
annual_income -0.079
interaction_count 0.044
lead_score 0.001


In [13]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    acc = train_and_score(features, C=C)
    print(C, round(acc, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
